In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==============================================================================
# 1. GLOBAL CONFIGURATION & MAPPING
# ==============================================================================
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ENTITY_MAPPING = {
    "Czechia": "Czech Republic",
    "Egypt, Arab Rep.": "Egypt",
    "Iran, Islamic Rep.": "Iran",
    "Korea, Rep.": "South Korea",
    "Korea, Republic of": "South Korea",
    "Korea": "South Korea",
    "Russian Federation": "Russia",
    "Slovak Republic": "Slovakia",
    "Turkiye": "Turkey",
    "Türkiye, Republic of": "Turkey",
    "China, People's Republic of": "China",
    "USA": "United States",
    "US": "United States"
}

ARTIFACTS = [
    "©IMF, 2026", 
    "Data from database: World Development Indicators", 
    "Last Updated: 04/08/2026",
    "nan", "NaN"
]

DEFICIENT_COUNTRIES = ['Mexico', 'Iran', 'Ethiopia', 'New Zealand', 'South Africa', 'Saudi Arabia']

# ==============================================================================
# 2. CORE PROCESSING FUNCTIONS
# ==============================================================================
def standardize_entities(df, col_name):
    """Cleans entity names, removes artifacts, and applies global mapping."""
    df = df.dropna(subset=[col_name])
    df[col_name] = df[col_name].astype(str).str.strip()
    df = df[~df[col_name].isin(ARTIFACTS)]
    df[col_name] = df[col_name].replace(ENTITY_MAPPING)
    return df

def process_world_bank(filepath, val_name='Value'):
    """Melts World Bank wide-format matrices into strict panel (long) format."""
    df = pd.read_csv(filepath)
    df = standardize_entities(df, 'Country Name')
    
    id_vars = ['Country Name', 'Country Code', 'Series Name', 'Series Code']
    year_cols = [c for c in df.columns if '[YR' in c]
    
    melted = pd.melt(df, id_vars=id_vars, value_vars=year_cols, 
                     var_name='Year_Raw', value_name=val_name)
    
    melted['Year'] = melted['Year_Raw'].str[:4].astype(int)
    melted = melted[(melted['Year'] >= 1995) & (melted['Year'] <= 2022)]
    melted[val_name] = pd.to_numeric(melted[val_name], errors='coerce')
    melted = melted.dropna(subset=[val_name])
    
    final_df = melted.pivot_table(index=['Country Name', 'Year'], 
                                  columns='Series Name', 
                                  values=val_name).reset_index()
    final_df.columns.name = None 
    return final_df

def process_imf(filepath):
    """Melts raw IMF wide-format data into long format."""
    df = pd.read_csv(filepath)
    country_col = df.columns[0]
    
    df = standardize_entities(df, country_col)
    df = df.rename(columns={country_col: 'Country Name'})
    
    # IMF has years as columns. Isolate all columns that aren't the Country Name.
    year_cols = [c for c in df.columns if c != 'Country Name']
    
    # Melt into long format
    df = pd.melt(df, id_vars=['Country Name'], value_vars=year_cols, 
                 var_name='Year_Raw', value_name='General Government Debt (% of GDP)')
    
    # Safely extract 4-digit years using regex
    df['Year'] = df['Year_Raw'].astype(str).str.extract(r'(\d{4})').astype(float)
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(int)
    
    # Clean the metric data (IMF uses 'no data' strings)
    df['General Government Debt (% of GDP)'] = pd.to_numeric(df['General Government Debt (% of GDP)'], errors='coerce')
    df = df.dropna(subset=['General Government Debt (% of GDP)'])
    
    return df[(df['Year'] >= 1995) & (df['Year'] <= 2022)]

def process_wid(filepath, val_col_name):
    """Melts raw WID wide-format data into long format."""
    df = pd.read_csv(filepath, sep=';', skiprows=1) # Skip WID metadata header
    country_col = df.columns[0]
    percentile_col = df.columns[1] # WID includes a percentile column we must bypass
    
    df = standardize_entities(df, country_col)
    df = df.rename(columns={country_col: 'Country Name'})
    
    # Isolate all year columns
    year_cols = [c for c in df.columns if c not in ['Country Name', percentile_col]]
    
    # Melt into long format
    df = pd.melt(df, id_vars=['Country Name'], value_vars=year_cols, 
                 var_name='Year_Raw', value_name=val_col_name)
    
    # Extract year from complex WID headers using regex
    df['Year'] = df['Year_Raw'].astype(str).str.extract(r'(\d{4})').astype(float)
    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(int)
    
    df[val_col_name] = pd.to_numeric(df[val_col_name], errors='coerce')
    df = df.dropna(subset=[val_col_name])
    
    return df[(df['Year'] >= 1995) & (df['Year'] <= 2022)][['Country Name', 'Year', val_col_name]]

# ==============================================================================
# 3. PIPELINE EXECUTION
# ==============================================================================
print("1. Processing World Bank Core Macro Data...")
wb_brics = process_world_bank(RAW_DIR / "Main Data BRICS.csv")
wb_oecd = process_world_bank(RAW_DIR / "Main Data OECD.csv")
wb_master = pd.concat([wb_brics, wb_oecd], ignore_index=True)

print("2. Processing World Bank CPI (Inflation) Data...")
cpi_master = process_world_bank(RAW_DIR / "CPI All countries WB.csv", val_name='Inflation (Annual %)')
if 'Inflation, consumer prices (annual %)' in cpi_master.columns:
    cpi_master = cpi_master.rename(columns={'Inflation, consumer prices (annual %)': 'Inflation (Annual %)'})

print("3. Processing IMF Government Debt Data...")
imf_brics = process_imf(RAW_DIR / "BRICS General Government Debt % of GDP.csv")
imf_oecd = process_imf(RAW_DIR / "OECD General Government Debt % of GDP.csv")
imf_master = pd.concat([imf_brics, imf_oecd], ignore_index=True)

print("4. Processing WID Wealth Inequality Data...")
wid_top10_brics = process_wid(RAW_DIR / "WID 10% top Wealth BRICS.csv", 'Wealth Inequality (Top 10%)')
wid_top10_oecd  = process_wid(RAW_DIR / "WID 10% top Wealth OECD.csv", 'Wealth Inequality (Top 10%)')
wid_bot50_brics = process_wid(RAW_DIR / "WID 50% bottom Wealth BRICS.csv", 'Wealth Inequality (Bottom 50%)')
wid_bot50_oecd  = process_wid(RAW_DIR / "WID 50% bottom Wealth OECD.csv", 'Wealth Inequality (Bottom 50%)')

wid_master = pd.concat([wid_top10_brics, wid_top10_oecd, wid_bot50_brics, wid_bot50_oecd], ignore_index=True)
wid_master = wid_master.groupby(['Country Name', 'Year']).first().reset_index()

# ==============================================================================
# 4. MASTER MERGE & ECONOMETRIC FILTERING
# ==============================================================================
print("5. Merging Master Panel...")
master_panel = pd.merge(wb_master, cpi_master, on=['Country Name', 'Year'], how='outer')
master_panel = pd.merge(master_panel, wid_master, on=['Country Name', 'Year'], how='outer')
master_panel = pd.merge(master_panel, imf_master, on=['Country Name', 'Year'], how='outer')

print("6. Applying Econometric Constraints (Cleaning & Filtering)...")
if 'Central government debt, total (% of GDP)' in master_panel.columns:
    master_panel = master_panel.drop(columns=['Central government debt, total (% of GDP)'])

master_panel = master_panel[~master_panel['Country Name'].isin(DEFICIENT_COUNTRIES)]

metric_columns = master_panel.columns.drop(['Country Name', 'Year'])
master_panel = master_panel.dropna(subset=metric_columns, how='all')

master_panel = master_panel.drop_duplicates(subset=['Country Name', 'Year'])
master_panel = master_panel.sort_values(by=['Country Name', 'Year']).reset_index(drop=True)

output_path = PROCESSED_DIR / "master_panel_data.csv"
master_panel.to_csv(output_path, index=False)

print("\n==================================================")
print("=== MASTER PANEL SUCCESSFULLY COMPILED ===")
print(f"Final True Panel Shape: {master_panel.shape}")
print(f"Entities present: {master_panel['Country Name'].nunique()}")
print("Columns Processed:")
for col in master_panel.columns:
    print(f"  - {col}")
print("==================================================")

1. Processing World Bank Core Macro Data...
2. Processing World Bank CPI (Inflation) Data...
3. Processing IMF Government Debt Data...
4. Processing WID Wealth Inequality Data...
5. Merging Master Panel...
6. Applying Econometric Constraints (Cleaning & Filtering)...

=== MASTER PANEL SUCCESSFULLY COMPILED ===
Final True Panel Shape: (1204, 12)
Entities present: 43
Columns Processed:
  - Country Name
  - Year
  - Foreign direct investment, net inflows (% of GDP)
  - Foreign direct investment, net outflows (% of GDP)
  - GDP growth (annual %)
  - Gross capital formation (% of GDP)
  - Population growth (annual %)
  - Inflation (Annual %)
  - Wealth Inequality (Top 10%)
  - Wealth Inequality (Bottom 50%)
  - Year_Raw
  - General Government Debt (% of GDP)
